# ⚙️ [1/4] Check your Google Drive

        DiverScan-release
        │
        ├── data
        │        └── yourdata_folder
        │
        ├── projects
        │        └── yourproject_folder
        │                └── dataset (🟢 `dataset_folder`)
        │                        ├── settings_of_preprocessing.txt
        │                        ├── dataset_standardized_for_machine_learning.npz
        │                        ├── dataset_unstandardized_for_post_analysis.npz
        │                        └── ...
        │
        └── utils

# ⚙️ [2/4] Use a GPU

See the above menu. Select `Runtime` -> `Change runtime type` -> your favorite GPU -> `Save`.

In [ ]:
# Hit left ▶️ of this cell and run it!
CUDA = "cuda:0"

# ⚙️ [3/4] Authorize this notebook

In [ ]:
# Hit left ▶️ of this cell and run it!
# Mounting Google Drive is what Colab needs to reach your files. Elsewhere
# there is nothing to mount, so the notebook carries on with local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab; set the folder paths below to local ones.")


# ⚙️ [4/4] Load data and search architecture for DiverScan

1.   Input followings, then select `Runtime` -> `Run all`.

2.   It may take long time. Hours - days.

3.   If the execution is interrupted, set `resume_from_last_run` to ☑️ then select `Runtime` -> `Run all` again. Do not change other parameters.
⚠️ Please wait at least 30 minutes before resuming to ensure that all files have been saved to Google Drive.

4.   When the connection shuts down after all executions are complete, save and then close the notebook.

5.   Go to the `03_train_model` notebook.

In [ ]:
#@title ⚙️ 1. General settings

# skip_architecture_search = False #@param {type:"boolean"}

project_folder = "/content/drive/MyDrive/DiverScan-release/projects/fruitfly" #@param {type:"string"}

name_of_folder_to_save_result = "optuna" #@param {type:"string"}

number_of_optuna_trials = 100 #@param {type:"integer"}

task_type = "classification" #@param ["classification"]

number_of_attention_branches = 5 #@param {type:"integer"}

resume_from_last_run = True #@param {type:"boolean"}

use_preset_hyperparameters = False #@param {type:"boolean"}
if use_preset_hyperparameters:
    number_of_attention_branches = 5

#@markdown - ⚠️ Increasing the `number_of_optuna_trials` raises the likelihood of finding the optimal architecture for your model, but it also extends the computation time.
#@markdown - ⚠️ When set `resume_from_last_run` to ☑️, optuna tries to run additional trials of `number_of_optuna_trials`.
#@markdown - ⚠️ When `use_preset_hyperparameters` is checked, Optuna does not run. `number_of_attention_branches` is set to 5, and `number_of_optuna_trials` is ignored. The preset hyperparameters are automatically saved.
#@markdown - ⚠️ Too many number_of_attention_branches may cause out of memory error.

# project_folder is the folder 01_build_dataset made under DiverScan/projects.
from pathlib import Path
project_folder = Path(project_folder)
dataset_folder = str(project_folder / "dataset")


In [ ]:
#@title ⚙️ 2. Advanced settings

seed = 0 #@param {type:"integer"}

number_of_epochs_for_optuna_trial = 30 #@param {type:"integer"}

batch_size = 4 #@param {type:"integer"}

learning_rate = 0.001 #@param {type:"number"}
learning_rate = float(learning_rate)

userset_length_to_calculate_kernel_size = None #@param {type:"raw"}
if userset_length_to_calculate_kernel_size:
    userset_length_to_calculate_kernel_size = int(userset_length_to_calculate_kernel_size)

C_CE = 3.0 #@param {type:"number"}
C_CE = float(C_CE)

C_S = 1.0 #@param {type:"number"}
C_S = float(C_S)

threshold_of_attention_distribution = 0.8 #@param {type:"number"}
threshold_of_attention_distribution = float(threshold_of_attention_distribution)

threshold_of_loss_penalty = "Default" #@param {type:"raw"}
if not threshold_of_loss_penalty == "Default":
    threshold_of_loss_penalty = float(threshold_of_loss_penalty)

lambda_for_total_variation_regularization = "Default" #@param {type:"raw"}
if not lambda_for_total_variation_regularization == "Default":
    lambda_for_total_variation_regularization = float(lambda_for_total_variation_regularization)

lambda_for_feature_coherence = 0.0 #@param {type:"number"}
lambda_for_feature_coherence = float(lambda_for_feature_coherence)

feature_coherence_threshold_scale = 5.0 #@param {type:"number"}
feature_coherence_threshold_scale = float(feature_coherence_threshold_scale)

feature_coherence_sigmoid_sharpness = 5.0 #@param {type:"number"}
feature_coherence_sigmoid_sharpness = float(feature_coherence_sigmoid_sharpness)

#@markdown - `lambda_for_feature_coherence`: weight of the feature coherence penalty. Set 0 to disable it (default). The penalty is the attention-weighted variance of the features each attention layer receives, so a smaller value means one attention branch is looking at a segment that is homogeneous in feature space.
#@markdown - `feature_coherence_threshold_scale`: attention values below this multiple of the uniform weight are softly gated out before the variance is computed. 5 is the standard setting, 10 is the stricter one.
#@markdown - `feature_coherence_sigmoid_sharpness`: sharpness of that soft gate.
#@markdown - Kernel sizes will be optimized in 1 % - 10 % length of `userset_length_to_calculate_kernel_size`.
#@markdown - If you have hundreds CSV files/condition, you can increase `batch_size` to save the computation time. ex. `batch_size` = 32.

In [ ]:
#@title ⚙️ 3. Advanced settings - single mode
#@markdown The multi mode ignores followings.

conv1d_out_channels = 64 #@param {type:"integer"}

lstm_hidden_size = 64 #@param {type:"integer"}

In [ ]:
#@title ⚙️ 4. Advanced settings - multi mode
#@markdown The single mode ignores followings.

node_shuffle = False #@param {type:"boolean"}

stgcn_out_channels = 64 #@param {type:"integer"}

#@markdown - ⚠️ Uncheck `node_shuffle` when each node has its own identity, for example a male and a female in the `multi-animal` mode, or the `multi-bodyparts` mode, where each node is a specific body part. Otherwise, checking `node_shuffle` is recommended.

# Codes

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = str(seed)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = "expandable_segments:True"
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

In [ ]:
# Change current directory
os.chdir(dataset_folder)

# Read settings_of_preprocessing.txt to get the data type and the code snapshot.
loaded_values = {}
with open("settings_of_preprocessing.txt", "r") as f:
    for line in f:
        key, separator, value = line.rstrip("\n").partition(": ")
        if separator:
            loaded_values[key] = value

data_type = loaded_values["data_type"]
if data_type == "single":
    model_type = "single"
if data_type == "multi_animals":
    model_type = "multi"
if data_type == "multi_bodyparts":
    model_type = "multi"

# 01_build_dataset copied the code into the project as utils_YYYYMMDDHHMMSS.
# Projects made before that carry only the original path.
utils_snapshot = loaded_values.get("utils_snapshot")
if utils_snapshot:
    utils_folder = str(project_folder / utils_snapshot)
else:
    utils_folder = loaded_values["utils_folder"]
    print("No code snapshot in this project; using", utils_folder)


In [ ]:
!pip install optuna

In [ ]:
from pathlib import Path
import pickle
import gzip
import random
import numpy as np
import torch
import torch.nn as nn
import optuna
import logging
import joblib
import copy

In [ ]:
# Make the DiverScan utils importable.
import sys

if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import learning_utils as learning_utils_module

if model_type == "single":
    import single_model as model_module
elif model_type == "multi":
    import multi_model as model_module
else:
    raise ValueError(f'model_type must be "single" or "multi", got {model_type!r}')

# Used to record the notebook and environment of each run.
import run_record as run_record_module


In [ ]:
# Fix the random seeds, switch on deterministic mode, and pick the device.
# https://pytorch.org/docs/stable/notes/randomness.html
DEVICE = learning_utils_module.setup_determinism(seed, CUDA)


In [ ]:
# Change current directory
os.chdir(dataset_folder)

# Make loaders
with gzip.open("dataset_standardized_for_machine_learning.pkl", "rb") as file:
    loaded_data = pickle.load(file)
train_val_test_dictionary = loaded_data["train_val_test_dictionary"]

dataloaders, number_of_classes, number_of_channels, number_of_timestamps, number_of_nodes, number_of_dimensions_of_edge_weight, number_of_channels_of_pose, val_x, val_label, edge_index, val_edge_weight, val_x_pose, has_pose = learning_utils_module.load_data(
    "train_and_val",
    train_val_test_dictionary,
    model_type,
    batch_size,
    DEVICE,
)

train_loader, val_loader = dataloaders

In [ ]:
# Validate input
if task_type == "classification":
    task_type = "classification"

In [ ]:
# Create a folder to save data
project_dir = Path(dataset_folder).parent
optuna_folder = project_dir / name_of_folder_to_save_result

if not optuna_folder.exists():
    optuna_folder.mkdir(parents=True)
elif not resume_from_last_run:  # The folder is exist but you do not resume the last run
    counter = 1
    while optuna_folder.exists():
        optuna_folder = project_dir / f"{name_of_folder_to_save_result}_{counter}"
        counter += 1
    optuna_folder.mkdir(parents=True)

# Change current directory
os.chdir(optuna_folder)

In [ ]:
# Save settings as a txt file

learning_settings = {
    "model_type": model_type,
    "utils_folder": utils_folder,
    "utils_snapshot": utils_snapshot,
    "optuna_folder": optuna_folder,
    "dataset_folder": dataset_folder,
    "task_type": task_type,
    "number_of_attention_branches": number_of_attention_branches,
    "seed": seed,
    "number_of_epochs_for_optuna_trial": number_of_epochs_for_optuna_trial,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "userset_length_to_calculate_kernel_size": userset_length_to_calculate_kernel_size,
    "C_CE": C_CE,
    "C_S": C_S,
    "threshold_of_attention_distribution": threshold_of_attention_distribution,
    "threshold_of_loss_penalty": threshold_of_loss_penalty,
    "lambda_for_total_variation_regularization": lambda_for_total_variation_regularization,
    "lambda_for_feature_coherence": lambda_for_feature_coherence,
    "feature_coherence_threshold_scale": feature_coherence_threshold_scale,
    "feature_coherence_sigmoid_sharpness": feature_coherence_sigmoid_sharpness,
    "conv1d_out_channels": conv1d_out_channels,
    "lstm_hidden_size": lstm_hidden_size,
    "node_shuffle": node_shuffle,
    "stgcn_out_channels": stgcn_out_channels,
}

file_name_base = "learning_settings"
learning_file = optuna_folder / f"{file_name_base}.txt"

if learning_file.exists():
    counter = 1
    while learning_file.exists():
        learning_file = optuna_folder / f"{file_name_base}_{counter}.txt"
        counter += 1

with open(learning_file, "a") as file:
    for key, parameter in learning_settings.items():
        file.write(f"{key}: {parameter}\n")

In [ ]:
# Record this notebook and the environment used for this run.
# The notebook file to record, assuming the layout DiverScan/notebooks and
# DiverScan/projects/<project>. In Colab the running notebook is fetched from
# the frontend instead, so the file is only needed outside Colab.
run_record_module.save_run_record(
    utils_folder,
    optuna_folder,
    notebook_path=project_folder.parent.parent / "notebooks" / "02_search_architecture.ipynb",
)


In [ ]:
# Save optuna log as a txt file
# Logger configuration
logger = logging.getLogger("optuna")
logger.setLevel(logging.INFO)  # Output log messages of INFO level and above

# File handler configuration (append to the existing file)
file_handler = logging.FileHandler("optuna_log.txt", mode="a")  # Open in 'a' mode to append
file_handler.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)

# Adding the handler to the logger
logger.addHandler(file_handler)

In [ ]:
# Calculate some hyperparameters
threshold_of_loss_penalty, lambda_for_total_variation_regularization, penalty_weight_for_attention_distribution = learning_utils_module.calculate_hyperparameters(
    threshold_of_loss_penalty,
    lambda_for_total_variation_regularization,
    C_CE,
    C_S,
    number_of_classes,
    number_of_attention_branches,
)

# Bundle the loss hyperparameters and the validation tensors into config objects.
loss_config = learning_utils_module.LossConfig(
    task_type=task_type,
    number_of_attention_branches=number_of_attention_branches,
    C_CE=C_CE,
    C_S=C_S,
    threshold_of_loss_penalty=threshold_of_loss_penalty,
    lambda_for_total_variation_regularization=lambda_for_total_variation_regularization,
    penalty_weight_for_attention_distribution=penalty_weight_for_attention_distribution,
    threshold_of_attention_distribution=threshold_of_attention_distribution,
    lambda_for_feature_coherence=lambda_for_feature_coherence,
    feature_coherence_threshold_scale=feature_coherence_threshold_scale,
    feature_coherence_sigmoid_sharpness=feature_coherence_sigmoid_sharpness,
)

val_data = learning_utils_module.ValidationData(
    x=val_x,
    label=val_label,
    edge_weight=val_edge_weight,
    x_pose=val_x_pose,
)

model_spec = learning_utils_module.ModelSpec(
    model_type=model_type,
    number_of_classes=number_of_classes,
    number_of_channels=number_of_channels,
    number_of_timestamps=number_of_timestamps,
    number_of_attention_branches=number_of_attention_branches,
    number_of_channels_of_pose=number_of_channels_of_pose,
    number_of_nodes=number_of_nodes,
    number_of_dimensions_of_edge_weight=number_of_dimensions_of_edge_weight,
    edge_index=edge_index,
    conv1d_out_channels=conv1d_out_channels,
    lstm_hidden_size=lstm_hidden_size,
    stgcn_out_channels=stgcn_out_channels,
    node_shuffle=node_shuffle,
)


In [ ]:
# Used by the preset hyperparameters below.
ensure_odd = learning_utils_module.ensure_odd


In [ ]:
#@title Preset architecture (used when `use_preset_hyperparameters` is checked)
#@markdown Edit the values below to change the architecture that is used when you skip the search.
#@markdown `ensure_odd(number_of_timestamps, r)` turns a fraction `r` of the sequence length into an odd kernel size.

if model_type == "single":
    preset_hyperparameters = {
        "use_preset_hyperparameters": True,
        "dropout_probability": 0.3,
        "weight_decay": 0.001,
        "MaskedConv1dBlock_before_branching": 2,
        "temporal_kernel_size_before_branching": ensure_odd(number_of_timestamps, 0.05),
        "branch_0_MaskedConv1dBlock": 4,
        "branch_0_MaskedLSTMBlock": 4,
        "branch_0_attention_position": 1,
        "branch_0_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.05),
        "branch_1_MaskedConv1dBlock": 4,
        "branch_1_MaskedLSTMBlock": 2,
        "branch_1_attention_position": 1,
        "branch_1_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.01),
        "branch_2_MaskedConv1dBlock": 3,
        "branch_2_MaskedLSTMBlock": 2,
        "branch_2_attention_position": 1,
        "branch_2_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.08),
        "branch_3_MaskedConv1dBlock": 1,
        "branch_3_MaskedLSTMBlock": 4,
        "branch_3_attention_position": 0,
        "branch_3_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.03),
        "branch_4_MaskedConv1dBlock": 2,
        "branch_4_MaskedLSTMBlock": 1,
        "branch_4_attention_position": 0,
        "branch_4_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.08),
        "cross_entropy_losses": [0.0, 0.0, 0.0, 0.0, 0.0],
        "similarities": [0.0, 0.0, 0.0, 0.0, 0.0],
        "attention_distribution_penalty": 0.0,
        "macro_averaged_accuracies": [0.0, 0.0, 0.0, 0.0, 0.0],
    }

elif model_type == "multi":
    preset_hyperparameters = {
        "use_preset_hyperparameters": True,
        "dropout_probability": 0.3,
        "weight_decay": 0.001,
        "STGCNBlock_before_branching": 3,
        "temporal_kernel_size_before_branching": ensure_odd(number_of_timestamps, 0.09),
        "branch_0_STGCNBlock": 7,
        "branch_0_attention_position": 7,
        "branch_0_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.07),
        "branch_1_STGCNBlock": 3,
        "branch_1_attention_position": 3,
        "branch_1_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.10),
        "branch_2_STGCNBlock": 0,
        "branch_2_attention_position": 0,
        "branch_2_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.02),
        "branch_3_STGCNBlock": 0,
        "branch_3_attention_position": 0,
        "branch_3_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.05),
        "branch_4_STGCNBlock": 0,
        "branch_4_attention_position": 0,
        "branch_4_temporal_kernel_size": ensure_odd(number_of_timestamps, 0.09),
        "cross_entropy_losses": [0.0, 0.0, 0.0, 0.0, 0.0],
        "similarities": [0.0, 0.0, 0.0, 0.0, 0.0],
        "attention_distribution_penalty": 0.0,
        "macro_averaged_accuracies": [0.0, 0.0, 0.0, 0.0, 0.0],
    }
else:
    raise ValueError(f'model_type must be "single" or "multi", got {model_type!r}')


In [ ]:
if use_preset_hyperparameters:

    # Skip the search and write a ready-made architecture to optuna_summary.txt.
    learning_utils_module.write_preset_hyperparameters(
        preset_hyperparameters,
        optuna_folder,
    )

else:
    # Optuna

    start_epoch = 0

    def objective(trial):

        # Suggest one architecture; the same call covers both model types.
        architecture = learning_utils_module.suggest_architecture(
            trial,
            model_spec,
            userset_length_to_calculate_kernel_size,
        )

        before_branch_config, branch_config = learning_utils_module.build_configs(
            model_spec,
            architecture["dropout_probability"],
            architecture["before_branching_blocks"],
            architecture["temporal_kernel_size_before_branching"],
            architecture["branch_list"],
        )

        model = learning_utils_module.build_model(
            model_module,
            model_spec,
            before_branch_config,
            branch_config,
            DEVICE,
        )

        # Set up the loss function and optimizer
        criterion = nn.CrossEntropyLoss().to(DEVICE)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=architecture["weight_decay"],
        )

        # Run training and validation
        val_loss, epoch, val_cross_entropy_losses, val_similarities, val_attention_distribution_penalty, val_macro_averaged_accuracies = learning_utils_module.train_and_validate(
            model_type,
            model,
            train_loader,
            val_loader,
            val_data,
            has_pose,
            criterion,
            optimizer,
            start_epoch,
            number_of_epochs_for_optuna_trial,
            loss_config,
            trial,  # Optuna trial
            save_interval_of_epochs=None,
        )

        trial.set_user_attr("cross_entropy_losses", val_cross_entropy_losses)
        trial.set_user_attr("similarities", val_similarities)
        trial.set_user_attr("attention_distribution_penalty", val_attention_distribution_penalty)
        trial.set_user_attr("macro_averaged_accuracies", val_macro_averaged_accuracies)

        # Intermediate report to Optuna.
        trial.report(val_loss, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        return val_loss


 # Search log (Optuna)

In [ ]:
#@title Log

if not use_preset_hyperparameters:
    # Resumes from study.pkl when it exists, and writes optuna_summary.txt.
    study = learning_utils_module.run_optuna_study(
        objective,
        number_of_optuna_trials,
        seed,
    )


try:
    from google.colab import runtime
    import time
    time.sleep(5)
    runtime.unassign()
    print("The runtime has been unassigned.")
except ImportError:
    # Not running in Google Colab environment
    pass